# TrainPilot quickstart

Runtime → Change runtime type → **T4 GPU**.

This notebook installs TrainPilot from GitHub, trains a tiny model on synthetic tensors (no dataset download), and prints a one-page report. It should finish in 1–2 minutes. TrainPilot never edits your script.

In [ ]:
%pip install -q git+https://github.com/linee64/TrainPilot.git

In [ ]:
import torch
from torch import nn
from trainpilot import compare_mixed_precision, diagnose, measure_batches, render_report

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device if device == "cpu" else torch.cuda.get_device_name(0))
torch.manual_seed(0)

warmup, steps, batch_size = 5, 15, 32
n = (warmup + steps) * batch_size

def make_loader():
    data = torch.utils.data.TensorDataset(
        torch.randn(n, 3, 32, 32),
        torch.randint(0, 10, (n,)),
    )
    return torch.utils.data.DataLoader(data, batch_size=batch_size, num_workers=0)

model = nn.Sequential(
    nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
    nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(32, 10),
).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
ce = nn.CrossEntropyLoss()

def step(batch):
    x, y = batch[0].to(device), batch[1].to(device)
    optimizer.zero_grad(set_to_none=True)
    loss = ce(model(x), y)
    loss.backward()
    optimizer.step()
    return int(x.shape[0])

def loss_fn(candidate, batch):
    x, y = batch[0].to(device), batch[1].to(device)
    return ce(candidate(x), y)

diagnosis = diagnose(measure_batches(make_loader(), step, warmup=warmup, steps=steps))
experiment = compare_mixed_precision(
    model, optimizer, make_loader(), loss_fn,
    seed=0, warmup=warmup, steps=steps, try_compile=False,
)
print(render_report(diagnosis, experiment))